In [78]:
# Hücre 1 — Efe Otopark Sistemleri (koridordan rota, final)
# • 3 (sol) | şerit | 2 (sağ) yerleşim, satırlar arası şerit
# • “Giriş / Gişe” rozeti orta şerit üzerinde
# • Rota: koridordan gider (karelerin içinden geçmez), son adımda kısa giriş
# • Uzaklık: otomatik km/m (örn. 1 km 500 m)
# • Yorum paneli: "// burada ..." şeklinde sunum notları

html_code = '''<!DOCTYPE html><html lang="tr"><head>
<meta charset="UTF-8"/><meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Efe Otopark Sistemleri — Sensörlü Simülasyon</title>
<style>
  :root{--bg:#0f172a;--card:#111827;--muted:#94a3b8;--ok:#22c55e;--bad:#ef4444;--route:#60a5fa;--text:#e5e7eb;--ring:#334155;
        --cell:110px;--gap:12px;--lane:36px;--c-comment:#14b8a6}
  *{box-sizing:border-box}
  body{margin:0;font-family:system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:radial-gradient(1200px 800px at 20% 0%,#0b1224,var(--bg));
       color:var(--text);display:grid;gap:16px;padding:24px;place-items:start center}
  header{width:min(1400px,95vw);display:flex;align-items:center;gap:14px}
  .brand{display:flex;align-items:center;gap:10px}
  .brand-name{font-weight:800;font-size:clamp(20px,3.6vw,30px)}
  .tagline{color:var(--muted);font-size:14px;margin-top:2px}
  .wrap{width:min(1400px,95vw);display:grid;grid-template-columns:320px 1fr;gap:16px}
  .card{background:linear-gradient(180deg,#0b1020 0%,var(--card) 100%);border:1px solid var(--ring);border-radius:14px;padding:14px;box-shadow:0 8px 30px rgba(0,0,0,.25)}
  .controls label{display:block;margin-top:8px;font-size:14px}
  .controls select,.controls button{width:100%;margin-top:6px;padding:12px 14px;border-radius:12px;border:1px solid var(--ring);
    background:#0b1224;color:var(--text);font-weight:600;transition:.15s}
  .controls .btns{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-top:10px}
  .btn:active{transform:scale(.98)}.btn.active{border-color:var(--route);box-shadow:0 0 0 3px rgba(96,165,250,.25) inset}
  #stats{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-top:12px}
  .stat{background:#0b1224;border:1px dashed var(--ring);border-radius:12px;padding:12px;text-align:center}.stat .v{font-size:22px;font-weight:800}
  .entranceBar{display:flex;justify-content:space-between;align-items:center;gap:10px;margin:6px 2px 10px}
  .chip{background:#0b1224;border:1px solid var(--ring);border-radius:16px;padding:5px 12px;font-size:12px;opacity:.95}

  /* — Otopark alanı — */
  .lotArea{position:relative;width:max-content;margin:0 auto}
  #parking-lot{
    display:grid;
    grid-template-columns: repeat(3,var(--cell)) var(--lane) repeat(2,var(--cell));
    column-gap: var(--gap);
    row-gap: var(--lane);
    grid-auto-rows: var(--cell);
    align-items:center;
  }
  #routeSVG{position:absolute;inset:0;pointer-events:none}
  #routeSVG .path{fill:none;stroke:var(--route);stroke-width:6;stroke-linecap:round;stroke-linejoin:round;filter:drop-shadow(0 0 6px rgba(96,165,250,.35))}
  #routeSVG .dot{fill:var(--route)}

  .spot{width:var(--cell);height:var(--cell);display:grid;place-items:center;border:1px solid var(--ring);border-radius:12px;
        background:rgba(34,197,94,.08);text-align:center;padding:10px}
  .spot small{color:var(--muted);font-size:12px}.spot .plate{font-weight:800;font-size:14px}
  .spot.ok{background:rgba(34,197,94,.12)}.spot.busy{background:rgba(239,68,68,.16);border-color:rgba(239,68,68,.4)}
  .spot.nearest{outline:2px solid var(--route);box-shadow:0 0 0 5px rgba(96,165,250,.25) inset}

  #available-spots{margin-top:8px} #directions{margin-top:4px}
  #log{max-height:320px;overflow:auto;font-size:13px;line-height:1.35} #log .entry{padding:6px 0;border-bottom:1px dashed var(--ring)}

  /* Yorum paneli */
  details.code-notes{margin-top:10px}
  .code-note{background:rgba(20,184,166,.12);border:1px solid rgba(20,184,166,.35);color:#a7f3d0;padding:8px 10px;border-radius:10px;
             font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:13px;margin:6px 0}
  .code-note b{color:var(--c-comment)}

  /* Giriş / Gişe rozeti (orta şerit üzerinde) */
  .gate-badge{position:absolute;padding:4px 10px;font-size:12px;background:#0b1224;border:1px solid var(--ring);border-radius:12px;
              transform:translate(-50%,-100%);color:var(--muted)}
  .gate-badge.bottom{transform:translate(-50%,0%)}
  footer{width:min(1400px,95vw);display:flex;justify-content:space-between;align-items:center;color:var(--muted);font-size:12px}
</style>
</head>
<body>
<header>
  <div class="brand">
    <!-- Basit SVG logo -->
    <svg width="44" height="44" viewBox="0 0 44 44" aria-label="Efe Otopark Logo">
      <defs><linearGradient id="g" x1="0" x2="1"><stop offset="0%" stop-color="#60a5fa"/><stop offset="100%" stop-color="#22c55e"/></linearGradient></defs>
      <rect x="2" y="2" width="40" height="40" rx="10" fill="#0b1224" stroke="#334155"/>
      <path d="M14 32 V12 h9 q7 0 7 6 q0 6 -7 6 h-5 v8 z M18 20 h5 q3 0 3 -2 q0 -2 -3 -2 h-5 v4 z" fill="url(#g)"/>
      <path d="M29 10 q6 3 6 9" fill="none" stroke="#60a5fa" stroke-width="2" stroke-linecap="round"/>
      <path d="M29 14 q4 2 4 5" fill="none" stroke="#60a5fa" stroke-width="2" stroke-linecap="round"/>
      <circle cx="29" cy="18" r="1.8" fill="#60a5fa"/>
    </svg>
    <div>
      <div class="brand-name">Efe Otopark Sistemleri</div>
      <div class="tagline">⚖️ sensör • 📡 ağ/geçit • ☁️ sunucu • 🧭 yönlendirme • <b>kalitenin tek adresi</b></div>
    </div>
  </div>
</header>

<div class="wrap">
  <!-- Sol panel -->
  <section class="card controls">
    <label>Giriş noktası (Kapı) 🚪
      <select id="gateSelect"><option value="north">Kuzey Kapı (üst orta)</option><option value="south">Güney Kapı (alt orta)</option></select>
    </label>
    <div class="btns"><button id="btnStart" class="btn">▶ Başlat (Çalışmıyor)</button><button id="btnPause" class="btn">Ⅱ Durdur</button></div>
    <div class="btns"><button id="btnRoute" class="btn">🧭 En Yakın Yolu Göster</button><button id="btnClearRoute" class="btn">Temizle</button></div>
    <div class="btns"><button id="btnReset" class="btn">♻️ Sıfırla</button><button id="btnExport" class="btn">⬇️ Günlük (CSV)</button></div>

    <div id="stats">
      <div class="stat"><div class="v" id="statFree">0</div><div class="k">Boş</div></div>
      <div class="stat"><div class="v" id="statBusy">0</div><div class="k">Dolu</div></div>
      <div class="stat"><div class="v" id="statTotal">0</div><div class="k">Toplam</div></div>
    </div>

    <details class="code-notes">
      <summary>📘 Proje Açıklamaları (yorumlar)</summary>
      <div class="code-note"><b>// burada</b> şemayı <b>3 park | şerit | 2 park</b> yaptım; satırlar arası da şerit bıraktım. 🚧</div>
      <div class="code-note"><b>// burada</b> “Giriş / Gişe” kontrol noktasını <b>orta şerit</b>e aldım. 🛂</div>
      <div class="code-note"><b>// burada</b> rota karelerin içinden değil <b>koridordan</b> geçiyor, en sonda kısa giriş yapıyor. 🧭</div>
      <div class="code-note"><b>// burada</b> uzaklığı blok * metre ile hesaplayıp <b>km/m</b> yazdırıyorum. 📏</div>
    </details>
  </section>

  <!-- Sağ: Otopark -->
  <section class="card">
    <div style="display:flex;justify-content:space-between;align-items:baseline;gap:10px;">
      <div id="time" class="muted">Saat: 07:00:00</div>
      <div id="nearestInfo" class="muted">En yakın boş yer: —</div>
    </div>
    <div class="entranceBar"><span class="chip">Kuzey Kapı</span><span class="chip">Güney Kapı</span></div>

    <div id="lotArea" class="lotArea">
      <svg id="routeSVG"></svg>
      <div id="gateBadge" class="gate-badge">Giriş / Gişe</div>
      <div id="parking-lot" aria-live="polite"></div>
    </div>

    <div id="available-spots" class="muted">Boş Yerler: yükleniyor…</div>
    <div id="directions" class="muted"></div>
    <hr style="border:none;border-top:1px solid var(--ring);margin:12px 0;"/>
    <div id="log" aria-label="Günlük"></div>
  </section>
</div>

<footer><div>Tek dosya sürümü • Yakınlık önceliği • “Giriş / Gişe” • SVG rota</div><div><span id="build"></span></div></footer>

<script>
  /* her blok kaç metre? 500 m */
  const METERS_PER_BLOCK = 500;

  const ROWS=4, COLS=5;
  const spots=Array.from({length:ROWS*COLS},(_,i)=>({id:i+1,status:'empty',plate:null,weightT:0,lastUpdate:Date.now()}));
  let simTimer=null, clockTimer=null, running=false, totalSeconds=0;

  const $ = s=>document.querySelector(s);
  function idxToRC(i){return{row:Math.floor(i/COLS),col:i%COLS}}
  function pad(n){return String(n).padStart(2,'0')}
  function nowGate(){const v=$('#gateSelect').value;return v==='north'?{row:-1,col:Math.floor(COLS/2),label:'Kuzey'}:{row:ROWS,col:Math.floor(COLS/2),label:'Güney'}}

  function randPlate(){const r=String(Math.floor(Math.random()*81)+1).padStart(2,'0');const L="ABCÇDEFGĞHIİJKLMNOÖPRSŞTUÜVYZ";
    let m="";for(let k=0;k<(Math.random()<.5?2:3);k++)m+=L[Math.floor(Math.random()*L.length)];
    const d=Math.random()<.5?String(Math.floor(100+Math.random()*900)):String(Math.floor(1000+Math.random()*9000));
    return `${r} ${m} ${d}`}

  function formatDistance(blocks){
    const m = blocks * METERS_PER_BLOCK;
    if (m < 1000) return `${m} m`;
    const km = Math.floor(m/1000), rem = m % 1000;
    return rem===0 ? `${km} km` : `${km} km ${rem} m`;
  }

  /* rozet her zaman orta şerit merkezinde */
  function placeGateBadge(w,h,cell,gap,lane){
    const g=nowGate();
    const laneX = 3*cell + 3*gap + lane/2;
    const b=$('#gateBadge');
    b.style.left=`${laneX}px`;
    if(g.row<0){ b.classList.remove('bottom'); b.style.top='-8px'; }
    else{ b.classList.add('bottom'); b.style.top=`${h+8}px`; }
  }

  function render(){
    const lot=$('#parking-lot'); lot.innerHTML=''; let free=0,busy=0;
    for(let i=0;i<spots.length;i++){
      const r=i/COLS|0, c=i%COLS, d=document.createElement('div');
      d.className='spot '+(spots[i].status==='empty'?'ok':'busy');
      const gcol=(c<3)?(c+1):(c+2); d.style.gridColumn=gcol; d.style.gridRow=r+1; d.dataset.index=i;
      d.innerHTML=`<small>Yer ${spots[i].id}</small><div class="plate">${spots[i].status==='empty'?'Boş':'Dolu'}</div>`;
      lot.appendChild(d); if(spots[i].status==='empty') free++; else busy++;
    }
    $('#available-spots').textContent = free ? `Boş Yerler: ${spots.map((s,ix)=>s.status==='empty'?ix+1:null).filter(Boolean).join(', ')}` : 'Boş Yer yok';

    const cell=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--cell'))||110;
    const gap=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--gap'))||12;
    const lane=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--lane'))||36;
    const w=5*cell+5*gap+lane, h=4*cell+3*lane;
    const svg=$('#routeSVG'); svg.setAttribute('width',w); svg.setAttribute('height',h); svg.innerHTML='';
    placeGateBadge(w,h,cell,gap,lane);
    highlightNearest();
    $('#statFree').textContent=free; $('#statBusy').textContent=busy; $('#statTotal').textContent=spots.length;
  }

  function findNearestFree(){
    const g=nowGate(); let best=null;
    for(let i=0;i<spots.length;i++){
      if(spots[i].status==='empty'){
        const r=i/COLS|0, c=i%COLS;
        const vertical=g.row<0?(r+1):(ROWS-r);       // girişten hedef satıra
        const horizontal=Math.abs(c-Math.floor(COLS/2)); // orta şeritten hedef kolona
        const blocks=vertical+horizontal;
        if(!best || blocks<best.blocks) best={idx:i,blocks};
      }
    } return best;
  }

  function clearRoute(){ $('#routeSVG').innerHTML=''; $('#directions').textContent=''; document.querySelectorAll('.spot.nearest').forEach(n=>n.classList.remove('nearest')); }

  // Rota: koridordan (boşluk) ilerler, en sonda kısa giriş yapar
  function drawRoute(toIdx){
    clearRoute(); if(toIdx==null) return;

    const g=nowGate();
    const cell=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--cell'))||110;
    const gap=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--gap'))||12;
    const lane=parseFloat(getComputedStyle(document.documentElement).getPropertyValue('--lane'))||36;
    const w=5*cell+5*gap+lane, h=4*cell+3*lane;

    const center=(r,c)=>{const x=(c<3)?(c*cell+c*gap+cell/2):(c*cell+(c+1)*gap+lane+cell/2); const y=r*cell+r*lane+cell/2; return {x,y}};
    const laneX = 3*cell + 3*gap + lane/2;                    // orta şerit merkez X
    const rowLaneY = k => (k+1)*cell + k*lane + lane/2;       // satır koridoru merkez Y (0..ROWS-2)

    const to = idxToRC(toIdx);
    const spotC = center(to.row, to.col);

    // kapıya göre en uygun satır koridoru (hedef satırın üstündeki ya da altındaki)
    const laneRowIdx = (g.row < 0) ? Math.max(0, to.row - 1) : Math.min(ROWS - 2, to.row);
    const entryY     = rowLaneY(laneRowIdx);
    const gateOutsideY = (g.row < 0) ? -lane*0.8 : h + lane*0.8;

    const pts = [
      {x: laneX,   y: gateOutsideY},  // Giriş / Gişe (ızgara dışı)
      {x: laneX,   y: entryY},        // orta şeritten içeri
      {x: spotC.x, y: entryY},        // satır koridorunda yatay
      {x: spotC.x, y: spotC.y}        // cebe kısa giriş
    ];

    const svg=$('#routeSVG');
    const poly=document.createElementNS('http://www.w3.org/2000/svg','polyline');
    poly.setAttribute('class','path'); poly.setAttribute('points', pts.map(p=>`${p.x},${p.y}`).join(' ')); svg.appendChild(poly);
    const dot=p=>{const e=document.createElementNS('http://www.w3.org/2000/svg','circle'); e.setAttribute('class','dot'); e.setAttribute('cx',p.x); e.setAttribute('cy',p.y); e.setAttribute('r',5); svg.appendChild(e)};
    dot(pts[0]); dot(pts[pts.length-1]);

    // metin: koridor yönleri + mesafe
    const blocks  = (g.row < 0 ? (to.row + 1) : (ROWS - to.row)) + Math.abs(to.col - Math.floor(COLS/2));
    const distTxt = formatDistance(blocks);
    const turnTxt = (to.col < 3) ? 'sola' : 'sağa';
    const vertTxt = (g.row < 0) ? 'aşağı' : 'yukarı';
    $('#directions').textContent = `🧭 Yol tarifi: Giriş / Gişe → ana koridordan ${vertTxt} → ${turnTxt} dön → Yer ${toIdx+1} • Uzaklık: ${distTxt}`;

    const target=document.querySelector(`.spot[data-index="${toIdx}"]`); if(target) target.classList.add('nearest');
  }

  function highlightNearest(){
    const best=findNearestFree(), lbl=$('#nearestInfo');
    if(!best){lbl.textContent='En yakın boş yer: yok'; clearRoute(); return;}
    lbl.textContent = `En yakın boş yer: Yer ${best.idx+1} • Uzaklık: ${formatDistance(best.blocks)}`;
    const el=document.querySelector(`.spot[data-index="${best.idx}"]`); if(el) el.classList.add('nearest');
  }

  function logEntry(type,idx,plate,wT){
    const t=$('#time').textContent.replace('Saat: ',''); const text=type==='in'?'🚗 Araç girdi':'🚙 Araç çıktı';
    const extra=type==='in'?` • ⚖️ ${wT.toFixed(1)} ton`:''; const e=document.createElement('div'); e.className='entry';
    e.textContent=`${t} • ${text} • Yer ${idx+1} (${plate||''})${extra}`; $('#log').appendChild(e); $('#log').scrollTop=$('#log').scrollHeight;
  }

  function tickClock(){ totalSeconds++; const h=7+Math.floor(totalSeconds/3600), m=Math.floor((totalSeconds%3600)/60), s=totalSeconds%60;
    $('#time').textContent=`Saat: ${pad(h)}:${pad(m)}:${pad(s)}` }

  function step(){
    const occ=spots.filter(s=>s.status==='occupied').length, ratio=occ/spots.length, preferEnter=(Math.random()<(0.65-0.3*ratio))||occ===0;
    if(preferEnter){ const best=findNearestFree(); if(best){ const i=best.idx,s=spots[i]; s.status='occupied'; s.plate=randPlate(); s.weightT=1+Math.random()*2; s.lastUpdate=Date.now(); logEntry('in',i,s.plate,s.weightT); } }
    else{ const occIdx=spots.map((s,i)=>s.status==='occupied'?i:-1).filter(i=>i>=0); if(occIdx.length){ const i=occIdx[Math.floor(Math.random()*occIdx.length)], s=spots[i]; const gone=s.plate; s.status='empty'; s.plate=null; s.weightT=0; s.lastUpdate=Date.now(); logEntry('out',i,gone,0); } }
    render(); simTimer=setTimeout(step,800+Math.floor(Math.random()*800));
  }

  function start(){ if(!running){ clockTimer=setInterval(tickClock,1000); step(); running=true; $('#btnStart').classList.add('active'); $('#btnStart').textContent='▶ Başlat (Çalışıyor)'; $('#btnPause').classList.remove('active'); } }
  function pause(){ clearTimeout(simTimer); simTimer=null; clearInterval(clockTimer); clockTimer=null; running=false; $('#btnStart').classList.remove('active'); $('#btnStart').textContent='▶ Başlat (Çalışmıyor)'; $('#btnPause').classList.add('active'); }
  function reset(){ pause(); totalSeconds=0; $('#time').textContent='Saat: 07:00:00'; spots.forEach(s=>{s.status='empty';s.plate=null;s.weightT=0;s.lastUpdate=Date.now()});
    $('#log').innerHTML=''; clearRoute(); render(); }
  function exportCSV(){ const header=['time','type','spot','plate']; const rows=[header.join(',')]; const blob=new Blob([rows.join('\\n')],{type:'text/csv;charset=utf-8;'}); const url=URL.createObjectURL(blob); const a=document.createElement('a'); a.href=url; a.download='parking_log.csv'; a.click(); URL.revokeObjectURL(url); }

  // olay bağlama + güvenli başlangıç
  document.addEventListener('DOMContentLoaded', ()=>{
    $('#btnStart').addEventListener('click',start);
    $('#btnPause').addEventListener('click',pause);
    $('#btnReset').addEventListener('click',reset);
    $('#btnExport').addEventListener('click',exportCSV);
    $('#btnClearRoute').addEventListener('click',clearRoute);
    $('#btnRoute').addEventListener('click',()=>{ pause(); const best=findNearestFree(); if(best) drawRoute(best.idx); });
    $('#gateSelect').addEventListener('change',()=>{ highlightNearest(); clearRoute(); render(); });

    document.getElementById('build').textContent=new Date().toLocaleString('tr-TR');
    try{ render(); start(); }catch(e){ console.error(e); alert('JS hatası: '+e.message); }
  });
</script>
</body></html>'''

with open("/content/index.html","w",encoding="utf-8") as f:
    f.write(html_code)

print("✅ index.html kaydedildi -> /content/index.html")


✅ index.html kaydedildi -> /content/index.html


In [79]:
# burada varsa eski sunucuyu kapatıyorum
!fuser -k 8000/tcp || true

# burada /content klasörünü 8000 portundan yayınlıyorum
!python3 -m http.server 8000 --directory /content >/dev/null 2>&1 &
print("✅ Sunucu 8000 portunda çalışıyor")


8000/tcp:            27168
✅ Sunucu 8000 portunda çalışıyor


In [80]:
# burada colab'in proxyPort fonksiyonunu kullanıp bana dışarıdan erişilebilir link verdiriyorum
from google.colab import output
base = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("🔗 Sunucu linki: ", base)


🔗 Sunucu linki:  https://8000-m-s-3egghukrfo1xs-a.asia-east1-0.prod.colab.dev


In [81]:
# burada linkin sonuna /index.html ekliyorum ki sayfam direk açılsın
base = output.eval_js("google.colab.kernel.proxyPort(8000)")
full = base.rstrip("/") + "/index.html"
print("👉 Açmak için tıkla:", full)


👉 Açmak için tıkla: https://8000-m-s-3egghukrfo1xs-a.asia-east1-0.prod.colab.dev/index.html


In [ ]:
# burada istersem sayfayı colab notebook içine iframe ile gömüp çalıştırabiliyorum
from IPython.display import IFrame
IFrame(full, width="100%", height=640)
